# SageMaker S3 Input / Output

Migrated from `terraform-glue-env-repo-2`. This notebook reads standardized Parquet output, builds dashboard-ready summaries, and writes them back to the CDCU data lake.

In [ ]:
%pip install -q pandas pyarrow boto3

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlparse
import json
import os
import shutil

import boto3
import pandas as pd

session = boto3.session.Session()
REGION = session.region_name or os.getenv("AWS_REGION", "ap-southeast-1")
account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
DATA_LAKE_BUCKET = os.getenv("CDCU_DATA_LAKE_BUCKET", f"cdcu-sit-data-lake-{account_id}-apse1")
SOURCE_S3_URI = os.getenv("CDCU_SOURCE_S3_URI", f"s3://{DATA_LAKE_BUCKET}/standardized/merged/")
TARGET_S3_URI = os.getenv("CDCU_TARGET_S3_URI", f"s3://{DATA_LAKE_BUCKET}/processed/dashboard/")

WORK_DIR = Path("/tmp/sagemaker-s3-input-output")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
s3 = boto3.client("s3", region_name=REGION)

In [ ]:
def parse_s3_uri(s3_uri):
    parsed = urlparse(s3_uri)
    if parsed.scheme != "s3" or not parsed.netloc:
        raise ValueError(f"Invalid S3 URI: {s3_uri}")
    return parsed.netloc, parsed.path.lstrip("/")


def list_parquet_files(s3_client, bucket, prefix):
    keys = []
    paginator = s3_client.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for item in page.get("Contents", []):
            if item["Key"].endswith(".parquet"):
                keys.append(item["Key"])
    return sorted(keys)


def upload_file(s3_client, local_path, target_s3_uri, filename):
    bucket, prefix = parse_s3_uri(target_s3_uri)
    key = f"{prefix.rstrip('/')}/{filename}"
    s3_client.upload_file(str(local_path), bucket, key)
    return f"s3://{bucket}/{key}"

In [ ]:
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

source_bucket, source_prefix = parse_s3_uri(SOURCE_S3_URI)
parquet_keys = list_parquet_files(s3, source_bucket, source_prefix)
print(f"Source: {SOURCE_S3_URI}")
print(f"Found {len(parquet_keys)} parquet file(s).")
if not parquet_keys:
    raise RuntimeError(f"No parquet files found under {SOURCE_S3_URI}")

local_files = []
for index, key in enumerate(parquet_keys, start=1):
    local_path = INPUT_DIR / f"part-{index:05d}.parquet"
    s3.download_file(source_bucket, key, str(local_path))
    local_files.append(local_path)

df = pd.concat([pd.read_parquet(path) for path in local_files], ignore_index=True)
print(f"Loaded {len(df)} row(s).")
df.head()

In [ ]:
generated_at = datetime.now(timezone.utc).isoformat()
dashboard_summary = pd.DataFrame([{
    "metric": "row_count",
    "value": int(len(df)),
    "generated_at_utc": generated_at,
}])
dashboard_summary

In [ ]:
outputs = {}
summary_csv = OUTPUT_DIR / "dashboard_summary.csv"
summary_parquet = OUTPUT_DIR / "dashboard_summary.parquet"
dashboard_summary.to_csv(summary_csv, index=False)
dashboard_summary.to_parquet(summary_parquet, index=False)
outputs[summary_csv.name] = summary_csv
outputs[summary_parquet.name] = summary_parquet

for column in ("status", "name"):
    if column in df.columns:
        counts = df.groupby(column, dropna=False).size().reset_index(name="record_count")
        csv_path = OUTPUT_DIR / f"{column}_counts.csv"
        parquet_path = OUTPUT_DIR / f"{column}_counts.parquet"
        counts.to_csv(csv_path, index=False)
        counts.to_parquet(parquet_path, index=False)
        outputs[csv_path.name] = csv_path
        outputs[parquet_path.name] = parquet_path

manifest = {
    "generated_at_utc": generated_at,
    "source_s3_uri": SOURCE_S3_URI,
    "target_s3_uri": TARGET_S3_URI,
    "source_rows": int(len(df)),
    "columns": list(df.columns),
    "files": sorted(outputs),
}
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
outputs[manifest_path.name] = manifest_path
outputs

In [ ]:
uploaded = {
    filename: upload_file(s3, local_path, TARGET_S3_URI, filename)
    for filename, local_path in outputs.items()
}
for filename, s3_uri in sorted(uploaded.items()):
    print(f"{filename}: {s3_uri}")